### Vector stores and retrievers
This video tutorial will familiarize you with LangChain's vector store and retriever abstractions. These abstractions are designed to support retrieval of data-- from (vector) databases and other sources-- for integration with LLM workflows. They are important for applications that fetch data to be reasoned over as part of model inference, as in the case of retrieval-augmented generation.

We will cover 
- Documents
- Vector stores
- Retrievers


In [ ]:
import pandas as pd
from langchain_core.documents import Document
from langchain_chroma import Chroma

In [ ]:
# !pip install langchain
# !pip install langchain-chroma
# !pip install langchain_groq

### Documents
LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has two attributes:

- page_content: a string representing the content;
- metadata: a dict containing arbitrary metadata.
The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual Document object often represents a chunk of a larger document.

Let's generate some sample documents:

In [ ]:
file=r"C:\Users\vikram.vadhirajan\Downloads\Mandatory_fill.xlsx"

In [ ]:
df = pd.read_excel(file)

In [ ]:
df

,Brand,SKU,Attribute,Product Type,Value_Combined
0,BPI,4000,Brake Hose Length,Brake Hose,19.4
1,BPI,4497,Brake Hose Length,Brake Hose,16.5
2,BPI,4751,Brake Hose Length,Brake Hose,22.3
3,BPI,4900,Brake Hose Length,Brake Hose,17.6
4,BPI,4960,Brake Hose Length,Brake Hose,24.7
...,...,...,...,...,...
39307,BPI,6840841,Overall Length (in ),Brake Hose,NaN
39308,BPI,382372-1,Overall Length (in ),Brake Hose,NaN
39309,BPI,382373-1,Overall Length (in ),Brake Hose,NaN
39310,BPI,382375-1,Overall Length (in ),Brake Hose,NaN


In [ ]:


columns = ['Brand', 'SKU', 'Attribute', 'Product Type', 'Value_Combined']

documents = []

for _, row in df[columns].iterrows():

    text = f"""
    Brand: {row['Brand']}
    SKU: {row['SKU']}
    Attribute: {row['Attribute']}
    Product Type: {row['Product Type']}
    Value: {row['Value_Combined']}
    """

    documents.append(Document(page_content=text))



In [ ]:
documents[:41000]

[Document(page_content='\n    Brand: BPI\n    SKU: 4000\n    Attribute: Brake Hose Length\n    Product Type: Brake Hose\n    Value: 19.4\n    '),
 Document(page_content='\n    Brand: BPI\n    SKU: 4497\n    Attribute: Brake Hose Length\n    Product Type: Brake Hose\n    Value: 16.5\n    '),
 Document(page_content='\n    Brand: BPI\n    SKU: 4751\n    Attribute: Brake Hose Length\n    Product Type: Brake Hose\n    Value: 22.3\n    '),
 Document(page_content='\n    Brand: BPI\n    SKU: 4900\n    Attribute: Brake Hose Length\n    Product Type: Brake Hose\n    Value: 17.6\n    '),
 Document(page_content='\n    Brand: BPI\n    SKU: 4960\n    Attribute: Brake Hose Length\n    Product Type: Brake Hose\n    Value: 24.7\n    '),
 Document(page_content='\n    Brand: BPI\n    SKU: 5433\n    Attribute: Brake Hose Length\n    Product Type: Brake Hose\n    Value: 12.6\n    '),
 Document(page_content='\n    Brand: BPI\n    SKU: 5784\n    Attribute: Brake Hose Length\n    Product Type: Brake Hose\n   

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq

groq_api_key=os.getenv("GROQ_API_KEY")

os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

llm=ChatGroq(groq_api_key=groq_api_key,model="openai/gpt-oss-120b")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000284B00A15E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000284B00A1AF0>, model_name='openai/gpt-oss-120b', groq_api_key=SecretStr('**********'))

In [ ]:
# !pip install langchain_huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
## VectorStores
from langchain_chroma import Chroma

vectorstore=Chroma.from_documents(documents[:41000],embedding=embeddings)
vectorstore


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
vectorstore.similarity_search("SKU: 11147")

[Document(page_content='\n    Brand: BPI\n    SKU: 48882110\n    Attribute: VMRS Code\n    Product Type: Brake Rotor\n    Value: 013-017-001\n    '),
 Document(page_content='\n    Brand: BPI\n    SKU: 48881033\n    Attribute: VMRS Code\n    Product Type: Brake Rotor\n    Value: 13017001\n    '),
 Document(page_content='\n    Brand: BPI\n    SKU: 48881011\n    Attribute: VMRS Code\n    Product Type: Brake Rotor\n    Value: 13017001\n    '),
 Document(page_content='\n    Brand: BPI\n    SKU: 48882111\n    Attribute: VMRS Code\n    Product Type: Brake Rotor\n    Value: 013-017-001\n    ')]

### Retrievers
LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method:

In [ ]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever=RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["11147"])

[[Document(page_content='\n    Brand: BPI\n    SKU: 6839901\n    Attribute: Overall Length (in )\n    Product Type: Brake Hose\n    Value: 9.800 in\n    ')]]

Vectorstores implement an as_retriever method that will generate a Retriever, specifically a VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the following:

In [ ]:
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["cat","dog"])


[[Document(page_content='\n    Brand: BPI\n    SKU: 609270\n    Attribute: Brand\n    Product Type: Brake Hose\n    Value: NAPA\n    ')],
 [Document(page_content='\n    Brand: BPI\n    SKU: 4886629\n    Attribute: VMRS Code\n    Product Type: Brake Rotor\n    Value: 13017001\n    ')]]

In [110]:
## RAG
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain={"context":retriever,"question":RunnablePassthrough()}|prompt|llm

response=rag_chain.invoke("tell me about SkU 381257")
print(response.content)


I cannot answer based on the provided context.
